In [1]:
!pip install -q roboflow transformers evaluate imageio albumentations opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 79.2 MB/s eta 0:00:00


In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="DjdMDpo1Hqf5EcYdNKev")

print("cracks data")
project_cracks = rf.workspace("abhinavs-workspace-uuyko").project("cracks-3ii36-nimar")
dataset_cracks = project_cracks.version(1).download("yolov8")
crack_dir = dataset_cracks.location

print("\ndrywall data")
project_drywall = rf.workspace("objectdetect-pu6rn").project("drywall-join-detect")
dataset_drywall = project_drywall.version(2).download("yolov8")
drywall_dir = dataset_drywall.location


cracks data
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to cracks-1 in yolov8:: 100%|██████████| 10743/10743 [00:03<00:00, 3262.73it/s]


drywall data
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to Drywall-Join-Detect-2 in yolov8:: 100%|██████████| 2053/2053 [00:00<00:00, 5206.05it/s]


In [3]:
import os
import glob
from PIL import Image
import cv2
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import numpy as np
import random
from transformers import CLIPSegProcessor
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

def yolo_to_mask(img_w, img_h, label_path):
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    if not os.path.exists(label_path):
        return mask

    with open(label_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5: continue
            coords = np.array(parts[1:], dtype=np.float32)
            pts = coords.reshape(-1, 2)
            pts[:, 0] *= img_w
            pts[:, 1] *= img_h
            pts = np.int32(pts)
            cv2.fillPoly(mask, [pts], 255)
    return mask

class Yolov8SegDataset(Dataset):
    def __init__(self, images_dir, labels_dir, prompts_list, processor, max_size=352):
        self.image_paths = glob.glob(os.path.join(images_dir, "*.jpg")) + \
                           glob.glob(os.path.join(images_dir, "*.png")) + \
                           glob.glob(os.path.join(images_dir, "*.jpeg"))
        self.labels_dir = labels_dir
        self.prompts_list = prompts_list
        self.processor = processor
        self.max_size = max_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        img_w, img_h = image.size

        basename = os.path.basename(img_path)
        name, _ = os.path.splitext(basename)
        label_path = os.path.join(self.labels_dir, name + ".txt")

        mask_arr = yolo_to_mask(img_w, img_h, label_path)
        mask = Image.fromarray(mask_arr)

        image = image.resize((self.max_size, self.max_size))
        mask = mask.resize((self.max_size, self.max_size), Image.NEAREST)

        mask_arr = np.array(mask)
        mask_tensor = torch.tensor((mask_arr > 127).astype(np.float32))

        prompt = random.choice(self.prompts_list)
        inputs = self.processor(text=prompt, images=image, padding="max_length", return_tensors="pt")

        for k,v in inputs.items():
            inputs[k] = v.squeeze(0)

        inputs['labels'] = mask_tensor
        return inputs, img_path, prompt

processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")

print(f"cracks directory initialized at: {crack_dir}")
print(f"drywall directory initialized at: {drywall_dir}")

try:
    train_crack_dataset = Yolov8SegDataset(
        os.path.join(crack_dir, "train/images"),
        os.path.join(crack_dir, "train/labels"),
        ["segment crack", "segment wall crack"],
        processor
    )

    train_drywall_dataset = Yolov8SegDataset(
        os.path.join(drywall_dir, "train/images"),
        os.path.join(drywall_dir, "train/labels"),
        ["segment taping area", "segment drywall seam"],
        processor
    )

    train_dataset = ConcatDataset([train_crack_dataset, train_drywall_dataset])
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    print(f"total images: {len(train_dataset)}")
except Exception as e:
    print(f"err: {e}")



The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


preprocessor_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/974 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

cracks directory initialized at: /content/cracks-1
drywall directory initialized at: /content/Drywall-Join-Detect-2
total images: 5984


## CLIPSeg


In [4]:
from transformers import CLIPSegForImageSegmentation
import torch.nn.functional as F
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")

for param in model.clip.parameters():
    param.requires_grad = False

model.to(device)
optimizer = optim.AdamW(model.decoder.parameters(), lr=5e-5)

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1 - ((2. * intersection + smooth) / (pred.sum() + target.sum() + smooth))

def bce_dice_loss(pred, target):
    bce = F.binary_cross_entropy_with_logits(pred, target)
    dice = dice_loss(pred, target)
    return bce + dice
print(f"model loaded to {device}!")

model.safetensors:   0%|          | 0.00/603M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

CLIPSegForImageSegmentation LOAD REPORT from: CIDAS/clipseg-rd64-refined
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
clip.vision_model.embeddings.position_ids | UNEXPECTED |  | 
clip.text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded to cuda!


## training logic



In [6]:
import time

epochs = 10
model.train()
for epoch in range(epochs):
    epoch_loss = 0
    start_time = time.time()
    for batch_idx, (inputs, _, _) in enumerate(train_loader):
        pixel_values = inputs["pixel_values"].to(device)
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)
        labels = inputs["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

        logits = outputs.logits
        logits = F.interpolate(logits.unsqueeze(1), size=labels.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)

        loss = bce_dice_loss(logits, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f} | Time: {time.time()-start_time:.2f}s")

torch.save(model.state_dict(), "clipseg_drywall_finetuned.pth")
print("training done nd weights saved")

Epoch 1/10 | Loss: 0.4844 | Time: 324.52s
Epoch 2/10 | Loss: 0.4638 | Time: 325.40s
Epoch 3/10 | Loss: 0.4461 | Time: 324.38s
Epoch 4/10 | Loss: 0.4350 | Time: 323.38s
Epoch 5/10 | Loss: 0.4185 | Time: 323.47s
Epoch 6/10 | Loss: 0.4091 | Time: 324.68s
Epoch 7/10 | Loss: 0.3969 | Time: 324.62s
Epoch 8/10 | Loss: 0.3887 | Time: 333.62s
Epoch 9/10 | Loss: 0.3777 | Time: 324.44s
Epoch 10/10 | Loss: 0.3676 | Time: 324.01s
training done nd weights saved


In [7]:
import shutil

# YOLOv8 uses "valid" directory for the validation/test split in Roboflow
test_crack_dataset = Yolov8SegDataset(
    os.path.join(crack_dir, "valid/images"),
    os.path.join(crack_dir, "valid/labels"),
    ["segment crack"],
    processor
)
test_drywall_dataset = Yolov8SegDataset(
    os.path.join(drywall_dir, "valid/images"),
    os.path.join(drywall_dir, "valid/labels"),
    ["segment taping area"],
    processor
)
test_dataset = ConcatDataset([test_crack_dataset, test_drywall_dataset])
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

def evaluate_and_predict(loader, threshold=0.5, save_dir="predictions"):
    model.eval()
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)
    os.makedirs(save_dir, exist_ok=True)
    total_dice = 0
    count = 0

    with torch.no_grad():
        for inputs, img_paths, prompts in loader:
            pixel_values = inputs["pixel_values"].to(device)
            input_ids = inputs["input_ids"].to(device)
            attention_mask = inputs["attention_mask"].to(device)
            labels = inputs["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
            logits = outputs.logits
            logits = F.interpolate(logits.unsqueeze(1), size=labels.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)

            preds = torch.sigmoid(logits) > threshold

            for i in range(len(preds)):
                pred_flat = preds[i].view(-1).float()
                target_flat = labels[i].view(-1).float()
                intersection = (pred_flat * target_flat).sum()
                dice = (2. * intersection) / (pred_flat.sum() + target_flat.sum() + 1e-8)
                total_dice += dice.item()
                count += 1

                orig_img = Image.open(img_paths[i])
                orig_w, orig_h = orig_img.size

                # Multiply by 255 to save strictly 0 and 255
                mask_img = Image.fromarray((preds[i].cpu().numpy() * 255).astype(np.uint8))
                mask_img = mask_img.resize((orig_w, orig_h), Image.NEAREST)

                basename = os.path.basename(img_paths[i]).split('.')[0]
                clean_prompt = prompts[i].replace(' ', '_')
                save_name = f"{basename}__{clean_prompt}.png"
                mask_img.save(os.path.join(save_dir, save_name))

    if count > 0:
        print(f"avg test dice score: {total_dice/count:.4f}")
    print(f"predicted binary masks saved to /content/{save_dir}/")

evaluate_and_predict(test_loader)


avg test dice score: 0.3298
predicted binary masks saved to /content/predictions/


In [8]:
!zip -r /content/submission_masks.zip /content/predictions
from google.colab import files
files.download("/content/submission_masks.zip")

  adding: content/predictions/ (stored 0%)
  adding: content/predictions/IMG_8254_JPG_jpg__segment_taping_area.png (deflated 83%)
  adding: content/predictions/crack400_jpg__segment_crack.png (deflated 31%)
  adding: content/predictions/3443_jpg__segment_crack.png (deflated 46%)
  adding: content/predictions/crack333_jpg__segment_crack.png (deflated 21%)
  adding: content/predictions/2170_jpg__segment_crack.png (deflated 38%)
  adding: content/predictions/302-dat_png_jpg__segment_crack.png (deflated 32%)
  adding: content/predictions/2000x1500_0_resized_jpg__segment_taping_area.png (deflated 83%)
  adding: content/predictions/a_31_56_png_jpg__segment_crack.png (deflated 9%)
  adding: content/predictions/616-dat_png_jpg__segment_crack.png (deflated 38%)
  adding: content/predictions/a_7_0_png_jpg__segment_crack.png (deflated 26%)
  adding: content/predictions/294-dat_png_jpg__segment_crack.png (deflated 44%)
  adding: content/predictions/crack267_jpg__segment_crack.png (deflated 16%)
  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>